# Режимы модели и управление градиентами

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Различать режим модели, запись графа и фактическое обучение параметров.

## Данные примера

После работы с пакетами вернёмся к предсказанию модели. Режим поведения слоёв и запись вычислений для нахождения производных включаются отдельно. Для проверки восстановим два знакомых объекта с ответами 5 и 4; вычисления остаются на CPU.

In [ ]:
import torch
X = torch.tensor([[2., 1.], [1., 3.]], dtype=torch.float64)
y = torch.tensor([5., 4.], dtype=torch.float64)


## Режимы модели

`torch.nn.Linear(2, 1)` — готовый слой: он принимает два признака и выдаёт число `w₁ * x₁ + w₂ * x₂ + b`. Здесь `w₁` и `w₂` — веса признаков, `b` — свободный член, то есть прибавляемая постоянная. Это объект модели с обучаемыми параметрами. `train()` включает режим обучения, `eval()` — режим оценки. Некоторые слои, которые изучим вместе с нейросетями, ведут себя в этих режимах по-разному. `eval()` сам по себе не отключает градиенты; `no_grad()` не переключает режим модели.

У слоя `Linear(d, q)` d — число входных признаков, q — число выходов. Библиотека хранит `weight` в форме `(q, d)`, то есть по строке на выход, и вычисляет `X @ weight.T + bias`. В [математике](../../../math/linear-algebra/matrices-products/) один вектор w записан столбцом; для одного выхода строка `weight[0]` содержит те же d коэффициентов. Разная запись осей не означает другую линейную модель.

Модель вызывается как функция: `model(X)` вычисляет предсказания. Их форма `(2, 1)` содержит лишнюю для сравнения с `y` последнюю ось длины 1. Метод `.squeeze(-1)` убирает именно эту ось и даёт `(2,)`; `-1` обозначает последнюю ось. Свойство `model.training` истинно в режиме обучения и ложно в режиме оценки.

Слой начинает со случайных весов. `torch.manual_seed(17)` задаёт начальное состояние генератора PyTorch перед его созданием, чтобы в том же окружении повторять инициализацию.

In [ ]:
torch.manual_seed(17)
model = torch.nn.Linear(2, 1).to(dtype=torch.float64, device="cpu")
model.train()
assert model.training
model.eval()
with torch.no_grad():
    output = model(X).squeeze(-1)
assert output.shape == y.shape
assert not output.requires_grad


Случайные веса этого слоя ещё не обучены: значение выхода не оценивает качество решения.

Для справки: [автоматическое дифференцирование PyTorch](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).

## Два независимых переключателя

`model.train()` и `model.eval()` задают режим поведения слоёв. `torch.no_grad()` управляет записью графа. Например, у линейного слоя результат одной формулы не зависит от режима, но его параметры всё равно требуют градиентов. В следующих нейросетях появятся слои, у которых обучение и оценка используют разные правила.

In [ ]:
model.eval()
tracked = model(X)
assert tracked.requires_grad
with torch.no_grad():
    untracked = model(X)
assert not untracked.requires_grad
assert torch.allclose(tracked, untracked)


Совпадение значений не означает одинакового расхода памяти: в первом случае записываются связи для обратного прохода. Для проверки качества обычно одновременно выбирают eval и отключают запись графа.

## Разобранный пример: размер результата

Два входных признака и один выход слоя означают матрицу ответов `(число_объектов, 1)`. `.squeeze(-1)` убирает только последнюю единичную ось. Это безопаснее общего `.squeeze()`, который при одном объекте может убрать и ось объектов, превратив ответ в скаляр.

## Один объект — всё ещё пакет

Проверим предупреждение про `squeeze` кодом: срез `X[:1]` сохраняет ось объектов, поэтому имеет форму `(1, 2)`.

In [ ]:
model.eval()
with torch.no_grad():
    one_column = model(X[:1])
assert one_column.shape == (1, 1)
assert one_column.squeeze().shape == ()
assert one_column.squeeze(-1).shape == (1,)


Форма `()` обозначает скаляр, у которого нет осей. В проверочном коде сохраняем `(1,)`, чтобы одинаково обрабатывать один объект и много объектов.

## Задачи

### 1. Базовая

Отключает ли eval вычисление градиентов?

<details>
<summary>Подсказка</summary>

Это переключатель режима слоёв.

</details>



### 2. Базовая

Переводит ли no_grad модель в режим оценки?

<details>
<summary>Подсказка</summary>

Запись графа и режим независимы.

</details>



### 3. Прикладная

Получите проверочные ответы без графа.

<details>
<summary>Подсказка</summary>

Сначала eval, затем контекст no_grad.

</details>



### 4. Прикладная

Верните модель в режим обучения.

<details>
<summary>Подсказка</summary>

Используйте train.

</details>



### 5. Олимпиадная

Чем опасен squeeze() без оси на ответе формы (1, 1)?

<details>
<summary>Подсказка</summary>

Обе оси единичные.

</details>



### 6. Олимпиадная

Доказывает ли изменение флага training, что веса обучились?

<details>
<summary>Подсказка</summary>

Режим не запускает обновление параметров.

</details>



## Итоги

Вы умеете отдельно управлять режимом модели и записью графа вычислений.

## Самопроверка

Какие два действия нужны для обычной итоговой проверки модели? Почему `train()` ещё не означает, что веса изменились? Как сохранить ось единственного объекта при удалении последней оси длины 1?

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

Нет.

### Задача 2

Нет.

### Задача 3

In [ ]:
model.eval()
with torch.no_grad():
    answers = model(X).squeeze(-1)
assert answers.shape == y.shape
assert not answers.requires_grad


### Задача 4

In [ ]:
model.train()
assert model.training


### Задача 5

Будут удалены обе оси, получится скаляр. squeeze(-1) сохранит ось объектов и даст (1,).

### Задача 6

Нет. Для обучения нужны вычисление ошибки, градиентов и шаг изменения весов.